# Horus ScriptMind — métricas e diagnóstico

Este notebook é somente leitura: ele consolida os artefatos produzidos pelos jobs SLURM sem iniciar anotação ou treinamento.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cwd = Path.cwd().resolve()
BACKEND = cwd.parent if cwd.name == 'notebooks' else (cwd / 'backend' if (cwd / 'backend').exists() else cwd)
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from scriptmind.paths import ScriptMindPaths
paths = ScriptMindPaths.defaults()
DATA = paths.data_root
RESULTS = paths.results_root
print('Data:', DATA)
print('Results:', RESULTS)

## 1. Inventário e isolamento

In [ ]:
inventory_path = DATA / 'inventory' / 'report.json'
inventory = json.loads(inventory_path.read_text(encoding='utf-8')) if inventory_path.exists() else {}
pd.json_normalize(inventory).T.rename(columns={0: 'value'})

## 2. Qualidade dos anotadores

In [ ]:
quality_path = DATA / 'inventory' / 'annotation_quality.json'
quality = json.loads(quality_path.read_text(encoding='utf-8')) if quality_path.exists() else {}
display(pd.DataFrame([quality]))
if quality:
    ax = pd.Series({k: quality.get(k) for k in ['cohen_kappa', 'exact_agreement', 'arbitration_rate']}).plot.bar(figsize=(7, 4), ylim=(0, 1), title='Concordância da anotação')
    ax.axhline(0.75, color='red', linestyle='--', label='κ mínimo')
    ax.legend(); plt.show()

## 3. CSID e transições do crime script

In [ ]:
csid_summary_path = DATA / 'csid' / 'summary.json'
csid_summary = json.loads(csid_summary_path.read_text(encoding='utf-8')) if csid_summary_path.exists() else {}
display(pd.json_normalize(csid_summary).T.rename(columns={0: 'value'}))
residual_path = DATA / 'csid' / 'transition_standardized_residuals.csv'
if residual_path.exists():
    residuals = pd.read_csv(residual_path, index_col=0)
    plt.figure(figsize=(13, 10))
    sns.heatmap(residuals, cmap='vlag', center=0, vmin=-4, vmax=4)
    plt.title('Resíduos padronizados das transições de intenção')
    plt.tight_layout(); plt.show()
    display(residuals.stack().rename('SR').loc[lambda s: s.abs() >= 1.96].sort_values(ascending=False).to_frame())

## 4. Métricas por modelo, seed e conjunto

In [ ]:
metrics_path = RESULTS / 'report' / 'scriptmind_metrics.csv'
metrics = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
display(metrics.sort_values(['dataset_name', 'model_name', 'f1_macro'], ascending=[True, True, False]) if not metrics.empty else metrics)

In [ ]:
core = ['f1_macro', 'f1_scam', 'precision_scam', 'recall_scam', 'roc_auc', 'pr_auc', 'FP', 'FN',
        'current_intent_macro_f1', 'next_intent_macro_f1', 'next_utterance_rouge_l',
        'next_utterance_bertscore_f1', 'next_utterance_semantic_cosine',
        'valid_json_rate', 'evidence_valid_rate', 'mean_prompt_tokens', 'mean_output_tokens']
available = [c for c in core if c in metrics.columns]
group_cols = [c for c in ['model_name', 'dataset_name'] if c in metrics.columns]
aggregate = metrics.groupby(group_cols)[available].agg(['mean', 'std', 'count']).round(4) if not metrics.empty and group_cols else pd.DataFrame()
display(aggregate)

In [ ]:
if not metrics.empty and {'model_name', 'dataset_name', 'f1_macro'} <= set(metrics.columns):
    plt.figure(figsize=(10, 5))
    sns.barplot(data=metrics, x='model_name', y='f1_macro', hue='dataset_name', errorbar='sd')
    plt.ylim(0, 1); plt.title('Detecção: macro-F1 por modelo e split')
    plt.xticks(rotation=20); plt.tight_layout(); plt.show()
task_metrics = [c for c in ['current_intent_macro_f1', 'next_intent_macro_f1', 'next_utterance_rouge_l'] if c in metrics.columns]
if task_metrics:
    melted = metrics.melt(id_vars=[c for c in ['model_name', 'dataset_name'] if c in metrics.columns], value_vars=task_metrics, var_name='task', value_name='score')
    plt.figure(figsize=(11, 5))
    sns.barplot(data=melted, x='task', y='score', hue='model_name', errorbar='sd')
    plt.ylim(0, 1); plt.title('Tarefas ScriptMind'); plt.tight_layout(); plt.show()

## 5. Ablations, validade estrutural e eficiência

In [ ]:
if not metrics.empty:
    config_cols = [c for c in ['ablation', 'origin_mode', 'mode', 'dataset_name', 'model_name'] if c in metrics.columns]
    display(metrics.groupby(config_cols)[available].mean(numeric_only=True).round(4) if config_cols else pd.DataFrame())
    validity = [c for c in ['valid_json_rate', 'evidence_valid_rate'] if c in metrics.columns]
    if validity:
        display(metrics.groupby([c for c in ['model_name', 'dataset_name'] if c in metrics.columns])[validity].mean().round(4))

## 6. LLM-as-a-Judge e validação humana

In [ ]:
judge_rows = []
for file in RESULTS.glob('evaluation/**/judge_summary.json'):
    judge_rows.append({**json.loads(file.read_text(encoding='utf-8')), 'path': str(file)})
display(pd.json_normalize(judge_rows) if judge_rows else pd.DataFrame())

## 7. Baselines Horus existentes (somente leitura)

In [ ]:
baseline_path = RESULTS / 'report' / 'existing_baselines.json'
baselines = json.loads(baseline_path.read_text(encoding='utf-8')) if baseline_path.exists() else []
display(pd.DataFrame(baselines))

## Interpretação

- Resultados das três seeds devem ser interpretados pela média, dispersão e IC de 95%, não apenas pelo melhor run.
- A validação externa mantém a prevalência original e é a referência de generalização.
- O LLM-as-a-Judge só pode virar métrica principal após correlação ≥ 0,70 com a auditoria humana.
- O estudo cognitivo com participantes é uma etapa posterior e não é inferido destas métricas computacionais.